In [ ]:
import pandas as pd
from src.data import MODELS_DIR, DATA_DIR, DATA_DIR_PROCESSED

from topic_gen.models import TRECTopic, Topics
from topic_gen.evaluate import CosineSimilarity, BertScore, JaccardIndex, TopicEvaluator, RelativeLength
import os

In [2]:
def load_topics_and_run_evaluation(
    reference_topics_ird, 
    regerated_topics_class, 
    regenerated_topics_path, 
    measures):
    
    # Load topics
    reference_topics = Topics.load_ird_topics(reference_topics_ird)
    generated_topics = Topics[regerated_topics_class].read_jsonl(
        regenerated_topics_path / "topics.jsonl")

    # Run evaluation
    results = TopicEvaluator.evaluate(
        predictions=generated_topics,
        references=reference_topics,
        measures=measures
    )

    # Present results
    scores = pd.DataFrame(results)
    scores["topic_id"] = scores["topic_id"].astype(str)
    scores = scores.merge(pd.DataFrame(reference_topics.topics).add_prefix(
        "ref_"), left_on="topic_id", right_on="ref_query_id")

    df = pd.DataFrame([row.model_dump()
                      for row in generated_topics.topics]).add_prefix("gen_")
    df["gen_topic_id"] = df["gen_topic_id"].astype(str)

    scores = scores.merge(df, left_on="topic_id", right_on="gen_topic_id")
    return scores

In [ ]:
measures = [
    CosineSimilarity(),
    BertScore(),
    JaccardIndex(),
    RelativeLength(),
]

for topic in os.listdir(DATA_DIR_PROCESSED / "topics"):
    topic_path = DATA_DIR_PROCESSED / "topics" / topic
    scores = load_topics_and_run_evaluation("disks45/nocr/trec-robust-2004", TRECTopic,
                                            topic_path, measures)
    break

OutOfMemoryError: CUDA out of memory. Tried to allocate 276.00 MiB. GPU 0 has a total capacity of 47.40 GiB of which 94.38 MiB is free. Process 2404621 has 40.80 GiB memory in use. Process 2769768 has 6.48 GiB memory in use. Of the allocated memory 5.80 GiB is allocated by PyTorch, and 388.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)